# 06 — Drawdown & Daily PnL

**V7**: All profit claims cite `net_pnl > 0` after fee+slippage deduction. No gross-spread claims.

Equity curve = cumsum of net_pnl sorted by fill_time.  
Drawdown = equity - running max equity.

In [ ]:
import sys
sys.path.insert(0, '..')

import pandas as pd
import matplotlib.pyplot as plt

In [ ]:
filled = pd.read_parquet('../data/raw/filled_pnl.parquet')

# Sort by fill_time (nanoseconds)
filled = filled.dropna(subset=['fill_time']).copy()
filled['fill_time'] = pd.to_numeric(filled['fill_time'])
filled = filled.sort_values('fill_time')
filled['ts'] = pd.to_datetime(filled['fill_time'], unit='ns', utc=True)

print(f'Trades with fill_time: {len(filled)}')
print(f'Date range: {filled["ts"].min()} → {filled["ts"].max()}')

In [ ]:
# Equity curve (cumulative net PnL)
filled['equity'] = filled['net_pnl'].cumsum()
filled['peak']   = filled['equity'].cummax()
filled['drawdown'] = filled['equity'] - filled['peak']

print(f'Final equity   : {filled["equity"].iloc[-1]:.6f}')
print(f'Max drawdown   : {filled["drawdown"].min():.6f}')
print(f'Profitable trades cited via net_pnl (V7): {(filled["net_pnl"] > 0).sum()} / {len(filled)}')

In [ ]:
fig, (ax1, ax2, ax3) = plt.subplots(3, 1, figsize=(12, 10), sharex=True)

# Equity curve
ax1.plot(filled['ts'], filled['equity'], color='steelblue')
ax1.set_title('Cumulative Net PnL (after fees + slippage)')
ax1.set_ylabel('Net PnL')
ax1.axhline(0, color='black', linewidth=0.5)

# Drawdown
ax2.fill_between(filled['ts'], filled['drawdown'], 0, color='red', alpha=0.4)
ax2.set_title('Drawdown')
ax2.set_ylabel('Drawdown')

# Daily PnL
daily = filled.set_index('ts')['net_pnl'].resample('D').sum()
colors = ['steelblue' if v >= 0 else 'red' for v in daily]
ax3.bar(daily.index, daily.values, color=colors, width=0.8)
ax3.set_title('Daily Net PnL')
ax3.set_ylabel('Net PnL')
ax3.axhline(0, color='black', linewidth=0.5)

plt.tight_layout()
plt.savefig('../data/raw/drawdown.png', dpi=120)
plt.show()